# S09 - Codificación de variables categóricas

Daniel Pallares

## Paso 1: Cargar el dataset escalado de la Semana 8

In [1]:
import pandas as pd

df = pd.read_csv("../S08_PD_PallaresArbelaezSanchez_DatasetEscalado.csv")
df.shape

(551790, 32)

## Paso 2: Identificar columnas categóricas y su cardinalidad

In [2]:
df.select_dtypes(include="object").nunique()

C:\Users\danie\AppData\Local\Temp\ipykernel_7488\251659734.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.select_dtypes(include="object").nunique()


id_pedido                    551790
id_cliente                   150637
fecha_pedido                    836
canal_compra                      4
metodo_pago                       4
ciudad_tienda                    12
categoria_producto                6
producto                         24
correo_cliente               460883
nivel_satisfaccion               15
nivel_lealtad                     4
comentario_cliente               13
fecha_actualizacion_stock      1177
dtype: int64

## Clasificación de mis columnas categóricas

| Columna | Tipo | Cardinalidad | Técnica |
|---|---|---|---|
| `canal_compra` | Nominal | Baja (4) | One-hot encoding |
| `metodo_pago` | Nominal | Baja (4) | One-hot encoding |
| `categoria_producto` | Nominal | Baja (6) | One-hot encoding |
| `nivel_lealtad` | Ordinal (Bronce < Plata < Oro < Platino) | Baja (4) | Ordinal encoding |
| `ciudad_tienda` | Nominal | Media/alta (12, y es la de mayor cardinalidad entre mis variables de categoría) | Target encoding |
| `producto` | Nominal | Media (24) | No se codifica: queda fuera del alcance de esta actividad; es redundante con `categoria_producto` y no la pide la guía |
| `nivel_satisfaccion` | Ordinal | Alta por inconsistencias de texto (15, por mayúsculas/espacios sin limpiar) | No se codifica esta semana: es un problema de calidad de datos ya señalado en `nivel_satisfaccion_imputado` desde semanas anteriores, no el foco de esta guía |
| `id_pedido`, `id_cliente`, `correo_cliente` | Identificadores | Muy alta (prácticamente todas distintas) | No se codifican: son llaves/identificadores, no variables predictoras |
| `fecha_pedido`, `fecha_actualizacion_stock`, `comentario_cliente` | Fecha / texto libre | - | No se codifican en esta guía |

## Paso 3: One-hot encoding de mis variables nominales de baja cardinalidad

In [3]:
from sklearn.preprocessing import OneHotEncoder

codificador_nominal = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
codificador_nominal.fit(df[["categoria_producto", "metodo_pago", "canal_compra"]])
codificador_nominal.get_feature_names_out()

array(['categoria_producto_Belleza', 'categoria_producto_Deportes',
       'categoria_producto_Electronica', 'categoria_producto_Hogar',
       'categoria_producto_Jugueteria', 'categoria_producto_Moda',
       'metodo_pago_Efectivo', 'metodo_pago_PayPal',
       'metodo_pago_Tarjeta', 'metodo_pago_Transferencia',
       'canal_compra_App', 'canal_compra_Marketplace',
       'canal_compra_Tienda', 'canal_compra_Web'], dtype=object)

Para transformar el dataframe completo uso `pd.get_dummies`, que es más simple de integrar con las demás columnas:

In [4]:
df_codificado = pd.get_dummies(
    df,
    columns=["categoria_producto", "metodo_pago", "canal_compra"],
    prefix=["cat", "pago", "canal"],
)
[c for c in df_codificado.columns if c.startswith(("cat_", "pago_", "canal_"))]

['cat_Belleza',
 'cat_Deportes',
 'cat_Electronica',
 'cat_Hogar',
 'cat_Jugueteria',
 'cat_Moda',
 'pago_Efectivo',
 'pago_PayPal',
 'pago_Tarjeta',
 'pago_Transferencia',
 'canal_App',
 'canal_Marketplace',
 'canal_Tienda',
 'canal_Web']

## Paso 4: Ordinal encoding de `nivel_lealtad`, con el orden correcto

In [5]:
orden_lealtad = ["Bronce", "Plata", "Oro", "Platino"]

from sklearn.preprocessing import OrdinalEncoder

codificador_lealtad = OrdinalEncoder(categories=[orden_lealtad])
df_codificado["nivel_lealtad_cod"] = codificador_lealtad.fit_transform(df_codificado[["nivel_lealtad"]])

df_codificado[["nivel_lealtad", "nivel_lealtad_cod"]].drop_duplicates().sort_values("nivel_lealtad_cod")

,nivel_lealtad,nivel_lealtad_cod
1,Bronce,0.0
3,Plata,1.0
5,Oro,2.0
0,Platino,3.0


## Por qué usé ordinal encoding en `nivel_lealtad`

`nivel_lealtad` tiene un orden real de negocio (Bronce < Plata < Oro < Platino: cada nivel representa más compromiso del cliente con el programa de lealtad que el anterior). Un one-hot encoding perdería esa relación de orden, mientras que el ordinal encoding la conserva en una sola columna numérica, siempre que se defina explícitamente la lista `orden_lealtad` en el orden real y no se deje que el codificador use el orden alfabético (que hubiera puesto Bronce=0, Oro=1, Plata=2, Platino=3, un orden incorrecto).

## Paso 5: Separar el dataset en entrenamiento y prueba

Antes de calcular el target encoding de `ciudad_tienda` divido el dataset, para no usar información de la prueba al ajustar el codificador (evitar fuga de datos).

In [6]:
from sklearn.model_selection import train_test_split

df_entrenamiento, df_prueba = train_test_split(df_codificado, test_size=0.2, random_state=42)
len(df_entrenamiento), len(df_prueba), len(df_codificado)

(441432, 110358, 551790)

## Paso 6: Target encoding de `ciudad_tienda`, ajustado solo con el entrenamiento

`ciudad_tienda` es mi variable de mayor cardinalidad entre las de categoría (12 ciudades). Reviso antes su distribución para decidir si hace falta agrupar categorías poco frecuentes:

In [7]:
df_entrenamiento["ciudad_tienda"].value_counts(dropna=False)

ciudad_tienda
Cartagena        35298
Bucaramanga      35189
Manizales        35156
Cali             35139
Pereira          35135
Cucuta           35121
Bogota           35039
Medellin         35039
Barranquilla     34979
Villavicencio    34946
Ibague           34940
Santa Marta      34779
NaN              20672
Name: count, dtype: int64

Las 12 ciudades están bien representadas (todas con miles de filas en el entrenamiento) y no hay categorías raras que distorsionen el promedio por grupo, así que no agrupo ninguna en "Otras": agruparlas aquí perdería información útil sin necesidad. `ciudad_tienda` sí tiene valores nulos (pedidos donde no se registró la ciudad); el `TargetEncoder` los trata como una categoría más al ajustar, así que no hace falta imputarlos antes de codificar.

In [8]:
from category_encoders import TargetEncoder

codificador_ciudad = TargetEncoder(cols=["ciudad_tienda"], smoothing=10)
codificador_ciudad.fit(df_entrenamiento["ciudad_tienda"], df_entrenamiento["monto_compra"])

,cols,['ciudad_tienda']
,verbose,0
,drop_invariant,False
,return_df,True
,handle_missing,'value'
,handle_unknown,'value'
,min_samples_leaf,20
,smoothing,10
,hierarchy,None
,min_group_size,None
,min_group_name,None


## Paso 7: Transformar ambos conjuntos con el mismo codificador ya ajustado

Nunca vuelvo a llamar `.fit()` sobre `df_prueba`: reutilizo el codificador ajustado solo con `df_entrenamiento` para transformar los dos conjuntos, así la prueba nunca influye en los valores aprendidos (esto es lo que evita la fuga de datos: el promedio de `monto_compra` por ciudad se calcula únicamente con las filas de entrenamiento).

In [9]:
df_entrenamiento["ciudad_tienda_cod"] = codificador_ciudad.transform(df_entrenamiento["ciudad_tienda"])
df_prueba["ciudad_tienda_cod"] = codificador_ciudad.transform(df_prueba["ciudad_tienda"])

df_entrenamiento["ciudad_tienda_cod"].describe()

count    441432.000000
mean     651731.697294
std        1712.545815
min      648696.047794
25%      650035.323070
50%      651986.395160
75%      652396.660118
max      654635.675460
Name: ciudad_tienda_cod, dtype: float64

In [10]:
df_prueba["ciudad_tienda_cod"].describe()

count    110358.000000
mean     651738.128108
std        1713.431410
min      648696.047794
25%      650035.323070
50%      651986.395160
75%      652396.660118
max      654635.675460
Name: ciudad_tienda_cod, dtype: float64

In [11]:
df_prueba["ciudad_tienda_cod"].isnull().sum()

np.int64(0)

## Justificación de las técnicas de codificación

- **One-hot (`categoria_producto`, `metodo_pago`, `canal_compra`)**: son nominales sin ningún orden entre categorías y de baja cardinalidad (4 a 6 valores), así que una columna binaria por categoría no infla demasiado el dataset y no inventa una relación de orden que no existe.
- **Ordinal (`nivel_lealtad`)**: tiene un orden real de negocio (Bronce < Plata < Oro < Platino), así que una sola columna numérica que respete ese orden es más eficiente que one-hot y conserva la información de progresión entre niveles.
- **Target encoding (`ciudad_tienda`)**: es mi variable con más categorías (12), y aunque no son cientos, un one-hot ya agregaría 12 columnas dispersas; el target encoding resume cada ciudad en un solo número relacionado con `monto_compra`, ajustado solo con el entrenamiento para no filtrar información de la prueba.

## Paso 8: Exportar el dataset final codificado

Uno de nuevo `df_entrenamiento` y `df_prueba` en un solo archivo, conservando las columnas categóricas originales junto a las columnas codificadas.

In [12]:
df_final = pd.concat([df_entrenamiento, df_prueba]).sort_index()
df_final.to_csv("S09_PD_PallaresArbelaezSanchez_DatasetCodificado.csv", index=False)
df_final.shape

(551790, 45)